In [29]:
import json
import pandas as pd

## Step 1: Load the raw dataset

The input file is `case.json`, stored inside the `datasets` folder.

We first load the file and inspect the number of records and the top-level structure.

In [30]:
with open("datasets/case.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total records:", len(data))
print("Top-level keys:", data[0].keys())

Total records: 37
Top-level keys: dict_keys(['EnqueuedTimeUtc', 'EventName', 'Payload'])


## Step 2: Inspect sample records

The `Payload` column is stored as a JSON string, not as a ready Python object.

So we parse it using `json.loads()` to understand the structure of each event type.

In [31]:
for row in data[:5]:
    payload_parsed = json.loads(row["Payload"])
    print("\nEventName:", row["EventName"])
    print("Parsed payload type:", type(payload_parsed))
    print("Parsed payload:", payload_parsed)


EventName: DynamicPrice_Result
Parsed payload type: <class 'dict'>
Parsed payload: {'provider': 'ApplyDynamicPriceRange', 'offerId': 'a6611d55-9624-4381-8cdd-323ee3689241', 'algorithmOutput': {'min_global': 85.0, 'min_recommended': 87.2, 'max_recommended': 97.65, 'differenceMinRecommendMinTheory': 2.2}}

EventName: DynamicPrice_Result
Parsed payload type: <class 'dict'>
Parsed payload: {'provider': 'ApplyDynamicPricePerOption', 'offerId': '56e0702c-0218-4626-8d3d-ae9d54b4503b', 'algorithmOutput': [{'uniqueOptionId': 'b0e296a9-0590-f0e0-8211-243a2ededb12', 'bestPrice': 92.45}, {'uniqueOptionId': 'd6562c24-0b37-5fb4-8275-65b7b8b47b87', 'bestPrice': 92.45}, {'uniqueOptionId': '8d0f9262-f543-d0c8-a869-33985ae3ecda', 'bestPrice': 92.45}, {'uniqueOptionId': '151e59ac-761a-96f5-d2b9-882037a9fd28', 'bestPrice': 94.6}, {'uniqueOptionId': '3cd346f4-d297-7568-2e50-d43a8e2fd0a9', 'bestPrice': 94.6}, {'uniqueOptionId': 'b7a7b6d1-4dae-7392-5aaf-f3369c29db1d', 'bestPrice': 93.0}, {'uniqueOptionId': 

## Step 3: Count event types

Before transforming the data, it is useful to profile it and understand how many records belong to each category.

We found three relevant groups:
- `CurateOffer_Result`
- `DynamicPrice_Result` with provider `ApplyDynamicPriceRange`
- `DynamicPrice_Result` with provider `ApplyDynamicPricePerOption`

In [32]:
curate_count = 0
range_count = 0
option_count = 0

for row in data:
    payload = json.loads(row["Payload"])
    
    if row["EventName"] == "CurateOffer_Result":
        curate_count += 1
    
    elif row["EventName"] == "DynamicPrice_Result":
        provider = payload["provider"]
        
        if provider == "ApplyDynamicPriceRange":
            range_count += 1
        elif provider == "ApplyDynamicPricePerOption":
            option_count += 1

print("CurateOffer_Result count:", curate_count)
print("DynamicPrice_Result with ApplyDynamicPriceRange count:", range_count)
print("DynamicPrice_Result with ApplyDynamicPricePerOption count:", option_count)

CurateOffer_Result count: 5
DynamicPrice_Result with ApplyDynamicPriceRange count: 25
DynamicPrice_Result with ApplyDynamicPricePerOption count: 7


## Step 4: Inspect the `CurateOffer_Result` structure

For `CurateOffer_Result`:
- the payload is a list of offers
- each offer contains an `options` list
- each option must become one final row in the output table

This means we need to flatten nested lists.

In [33]:
for row in data:
    if row["EventName"] == "CurateOffer_Result":
        payload = json.loads(row["Payload"])
        
        print("Type of payload:", type(payload))
        print("Number of offers in this payload:", len(payload))
        
        first_offer = payload[0]
        print("\nKeys in first offer:")
        print(first_offer.keys())
        
        print("\nType of options:", type(first_offer["options"]))
        print("Number of options in first offer:", len(first_offer["options"]))
        
        print("\nKeys in first option:")
        print(first_offer["options"][0].keys())
        
        break

Type of payload: <class 'list'>
Number of offers in this payload: 4

Keys in first offer:
dict_keys(['curationProvider', 'offerId', 'dealerId', 'options'])

Type of options: <class 'list'>
Number of options in first offer: 2

Keys in first option:
dict_keys(['uniqueOptionId', 'optionId', 'isMobileDealer', 'isOpen', 'eta', 'chamaScore', 'productBrand', 'isWinner', 'minimumPrice', 'maximumPrice', 'dynamicPrice', 'finalPrice'])


## Step 5: Build `CuratedOfferOptions.csv`

Each row in the final table represents one option from the nested `options` list.

Fields come from three levels:
- outer row: `EnqueuedTimeUtc`
- offer level: `curationProvider`, `offerId`, `dealerId`
- option level: option-specific fields

In [34]:
curated_rows = []

for row in data:
    if row["EventName"] == "CurateOffer_Result":
        payload = json.loads(row["Payload"])
        enqueued_time = row["EnqueuedTimeUtc"]
        
        for offer in payload:
            for option in offer["options"]:
                curated_rows.append({
                    "CurationProvider": offer.get("curationProvider"),
                    "OfferId": offer.get("offerId"),
                    "DealerId": offer.get("dealerId"),
                    "UniqueOptionId": option.get("uniqueOptionId"),
                    "OptionId": option.get("optionId"),
                    "IsMobileDealer": option.get("isMobileDealer"),
                    "IsOpen": option.get("isOpen"),
                    "Eta": option.get("eta"),
                    "ChamaScore": option.get("chamaScore"),
                    "ProductBrand": option.get("productBrand"),
                    "IsWinner": option.get("isWinner"),
                    "MinimumPrice": option.get("minimumPrice"),
                    "MaximumPrice": option.get("maximumPrice"),
                    "DynamicPrice": option.get("dynamicPrice"),
                    "FinalPrice": option.get("finalPrice"),
                    "DefeatPrimaryReason": option.get("defeatPrimaryReason"),
                    "DefeatReasons": option.get("defeatReasons"),
                    "EnqueuedTimeUtc": enqueued_time
                })

print("Total flattened curated rows:", len(curated_rows))
print(curated_rows[0])

Total flattened curated rows: 40
{'CurationProvider': 'ByPrice', 'OfferId': '149f0e53-ff85-425f-a01a-8710f06704ea', 'DealerId': '6517', 'UniqueOptionId': 'b0e296a9-0590-f0e0-8211-243a2ededb12', 'OptionId': '6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2', 'IsMobileDealer': True, 'IsOpen': True, 'Eta': '1:00', 'ChamaScore': 8.0, 'ProductBrand': 'ULTRAGAZ', 'IsWinner': True, 'MinimumPrice': 90.0, 'MaximumPrice': 180.0, 'DynamicPrice': 91.9, 'FinalPrice': 91.9, 'DefeatPrimaryReason': None, 'DefeatReasons': None, 'EnqueuedTimeUtc': '2021-08-25 05:02:55 UTC'}


## Step 6: Convert curated rows into a DataFrame

Now that the nested JSON has been flattened, we convert the rows into a tabular structure using Pandas.

In [35]:
df_curated = pd.DataFrame(curated_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print(df_curated.head())

  CurationProvider                               OfferId DealerId  \
0          ByPrice  149f0e53-ff85-425f-a01a-8710f06704ea     6517   
1          ByPrice  149f0e53-ff85-425f-a01a-8710f06704ea     6517   
2          ByPrice  149f0e53-ff85-425f-a01a-8710f06704ea     9047   
3          ByPrice  149f0e53-ff85-425f-a01a-8710f06704ea     9047   
4          ByPrice  149f0e53-ff85-425f-a01a-8710f06704ea     9047   

                         UniqueOptionId  \
0  b0e296a9-0590-f0e0-8211-243a2ededb12   
1  d6562c24-0b37-5fb4-8275-65b7b8b47b87   
2  8d0f9262-f543-d0c8-a869-33985ae3ecda   
3  3cd346f4-d297-7568-2e50-d43a8e2fd0a9   
4  577e4bbd-f49d-ac23-56a6-e70072a05229   

                                       OptionId  IsMobileDealer  IsOpen   Eta  \
0  6517 || dd839e4c-9f84-45eb-9cb2-9069fecf70f2            True    True  1:00   
1                                  6517 || 6517           False   False  0:01   
2                      9047 || 9047 || ULTRAGAZ           False   False  1:00   
3 

## Step 7: Convert UTC timestamp to Brazil timezone (UTC-3)

The assignment requires the output date format as:

`DD/MM/YYYY`

We convert the original UTC timestamp to UTC-3 and then format it as a date string.

In [36]:
df_curated["EnqueuedTimeUtc"] = pd.to_datetime(
    df_curated["EnqueuedTimeUtc"],
    format="%Y-%m-%d %H:%M:%S UTC"
)

df_curated["EnqueuedTimeSP"] = (
    df_curated["EnqueuedTimeUtc"] - pd.Timedelta(hours=3)
).dt.strftime("%d/%m/%Y")

print(df_curated[["EnqueuedTimeUtc", "EnqueuedTimeSP"]].head())

      EnqueuedTimeUtc EnqueuedTimeSP
0 2021-08-25 05:02:55     25/08/2021
1 2021-08-25 05:02:55     25/08/2021
2 2021-08-25 05:02:55     25/08/2021
3 2021-08-25 05:02:55     25/08/2021
4 2021-08-25 05:02:55     25/08/2021


## Step 8: Export `CuratedOfferOptions.csv`

We keep only the required output columns and save the result as a CSV file.

In [37]:
df_curated_final = df_curated[[
    "CurationProvider",
    "OfferId",
    "DealerId",
    "UniqueOptionId",
    "OptionId",
    "IsMobileDealer",
    "IsOpen",
    "Eta",
    "ChamaScore",
    "ProductBrand",
    "IsWinner",
    "MinimumPrice",
    "MaximumPrice",
    "DynamicPrice",
    "FinalPrice",
    "DefeatPrimaryReason",
    "DefeatReasons",
    "EnqueuedTimeSP"
]]

df_curated_final.to_csv("CuratedOfferOptions.csv", index=False)
print("CuratedOfferOptions.csv created!")

CuratedOfferOptions.csv created!


## Step 9: Inspect `DynamicPrice_Result` for per-option pricing

For provider `ApplyDynamicPricePerOption`:
- payload is a dictionary
- `algorithmOutput` is a list
- each item in that list becomes one output row

In [38]:
for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPricePerOption":
            print("Payload keys:", payload.keys())
            print("Type of algorithmOutput:", type(payload["algorithmOutput"]))
            print("Length of algorithmOutput:", len(payload["algorithmOutput"]))
            print("First item in algorithmOutput:")
            print(payload["algorithmOutput"][0])
            break

Payload keys: dict_keys(['provider', 'offerId', 'algorithmOutput'])
Type of algorithmOutput: <class 'list'>
Length of algorithmOutput: 8
First item in algorithmOutput:
{'uniqueOptionId': 'b0e296a9-0590-f0e0-8211-243a2ededb12', 'bestPrice': 92.45}


## Step 10: Build `DynamicPriceOption.csv`

Each row in this output represents one item inside the `algorithmOutput` list for per-option pricing events.

In [39]:
dynamic_option_rows = []

for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPricePerOption":
            enqueued_time = row["EnqueuedTimeUtc"]
            
            for item in payload["algorithmOutput"]:
                dynamic_option_rows.append({
                    "Provider": payload.get("provider"),
                    "OfferId": payload.get("offerId"),
                    "UniqueOptionId": item.get("uniqueOptionId"),
                    "BestPrice": item.get("bestPrice"),
                    "EnqueuedTimeUtc": enqueued_time
                })

print("Total dynamic option rows:", len(dynamic_option_rows))
print(dynamic_option_rows[0])

Total dynamic option rows: 56
{'Provider': 'ApplyDynamicPricePerOption', 'OfferId': '56e0702c-0218-4626-8d3d-ae9d54b4503b', 'UniqueOptionId': 'b0e296a9-0590-f0e0-8211-243a2ededb12', 'BestPrice': 92.45, 'EnqueuedTimeUtc': '2021-08-18 11:43:23 UTC'}


In [40]:
df_option = pd.DataFrame(dynamic_option_rows)
print(df_option.head())

                     Provider                               OfferId  \
0  ApplyDynamicPricePerOption  56e0702c-0218-4626-8d3d-ae9d54b4503b   
1  ApplyDynamicPricePerOption  56e0702c-0218-4626-8d3d-ae9d54b4503b   
2  ApplyDynamicPricePerOption  56e0702c-0218-4626-8d3d-ae9d54b4503b   
3  ApplyDynamicPricePerOption  56e0702c-0218-4626-8d3d-ae9d54b4503b   
4  ApplyDynamicPricePerOption  56e0702c-0218-4626-8d3d-ae9d54b4503b   

                         UniqueOptionId  BestPrice          EnqueuedTimeUtc  
0  b0e296a9-0590-f0e0-8211-243a2ededb12      92.45  2021-08-18 11:43:23 UTC  
1  d6562c24-0b37-5fb4-8275-65b7b8b47b87      92.45  2021-08-18 11:43:23 UTC  
2  8d0f9262-f543-d0c8-a869-33985ae3ecda      92.45  2021-08-18 11:43:23 UTC  
3  151e59ac-761a-96f5-d2b9-882037a9fd28      94.60  2021-08-18 11:43:23 UTC  
4  3cd346f4-d297-7568-2e50-d43a8e2fd0a9      94.60  2021-08-18 11:43:23 UTC  


In [41]:
df_option["EnqueuedTimeUtc"] = pd.to_datetime(
    df_option["EnqueuedTimeUtc"],
    format="%Y-%m-%d %H:%M:%S UTC"
)

df_option["EnqueuedTimeSP"] = (
    df_option["EnqueuedTimeUtc"] - pd.Timedelta(hours=3)
).dt.strftime("%d/%m/%Y")

In [42]:
df_option_final = df_option[[
    "Provider",
    "OfferId",
    "UniqueOptionId",
    "BestPrice",
    "EnqueuedTimeSP"
]]

df_option_final.to_csv("DynamicPriceOption.csv", index=False)
print("DynamicPriceOption.csv created!")

DynamicPriceOption.csv created!


## Step 11: Inspect `DynamicPrice_Result` for pricing range events

For provider `ApplyDynamicPriceRange`:
- payload is a dictionary
- `algorithmOutput` is also a dictionary
- each matching event becomes one output row

In [43]:
for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPriceRange":
            print("Payload keys:", payload.keys())
            print("Type of algorithmOutput:", type(payload["algorithmOutput"]))
            print("algorithmOutput contents:", payload["algorithmOutput"])
            break

Payload keys: dict_keys(['provider', 'offerId', 'algorithmOutput'])
Type of algorithmOutput: <class 'dict'>
algorithmOutput contents: {'min_global': 85.0, 'min_recommended': 87.2, 'max_recommended': 97.65, 'differenceMinRecommendMinTheory': 2.2}


## Step 12: Build `DynamicPriceRange.csv`

Unlike the previous case, `algorithmOutput` is a dictionary here, so each event produces one final row.

In [44]:
dynamic_range_rows = []

for row in data:
    if row["EventName"] == "DynamicPrice_Result":
        payload = json.loads(row["Payload"])
        
        if payload["provider"] == "ApplyDynamicPriceRange":
            algo = payload["algorithmOutput"]
            enqueued_time = row["EnqueuedTimeUtc"]
            
            dynamic_range_rows.append({
                "Provider": payload.get("provider"),
                "OfferId": payload.get("offerId"),
                "MinGlobal": algo.get("min_global"),
                "MinRecommended": algo.get("min_recommended"),
                "MaxRecommended": algo.get("max_recommended"),
                "DifferenceMinRecommendMinTheory": algo.get("differenceMinRecommendMinTheory"),
                "EnqueuedTimeUtc": enqueued_time
            })

print("Total dynamic range rows:", len(dynamic_range_rows))
print(dynamic_range_rows[0])

Total dynamic range rows: 25
{'Provider': 'ApplyDynamicPriceRange', 'OfferId': 'a6611d55-9624-4381-8cdd-323ee3689241', 'MinGlobal': 85.0, 'MinRecommended': 87.2, 'MaxRecommended': 97.65, 'DifferenceMinRecommendMinTheory': 2.2, 'EnqueuedTimeUtc': '2021-09-05 08:04:08 UTC'}


In [45]:
df_range = pd.DataFrame(dynamic_range_rows)
print(df_range.head())

                 Provider                               OfferId  MinGlobal  \
0  ApplyDynamicPriceRange  a6611d55-9624-4381-8cdd-323ee3689241      85.00   
1  ApplyDynamicPriceRange  b8c636fa-8241-47dc-ac40-bdf438a04d9c      85.00   
2  ApplyDynamicPriceRange  3d32f7fb-396d-4d3f-b673-dea1f7dc41b7      85.00   
3  ApplyDynamicPriceRange  329194f3-95a4-45ef-b3d0-2796f74ce2a0      85.00   
4  ApplyDynamicPriceRange  fdcfde5c-113d-4a59-9ae0-8bc31e2943d8      87.35   

   MinRecommended  MaxRecommended  DifferenceMinRecommendMinTheory  \
0           87.20           97.65                              2.2   
1           87.20           97.65                              2.2   
2           87.20           97.65                              2.2   
3           87.20           97.65                              2.2   
4           89.25           99.95                              1.9   

           EnqueuedTimeUtc  
0  2021-09-05 08:04:08 UTC  
1  2021-09-05 09:04:04 UTC  
2  2021-09-05 08:03:28 

In [46]:
df_range["EnqueuedTimeUtc"] = pd.to_datetime(
    df_range["EnqueuedTimeUtc"],
    format="%Y-%m-%d %H:%M:%S UTC"
)

df_range["EnqueuedTimeSP"] = (
    df_range["EnqueuedTimeUtc"] - pd.Timedelta(hours=3)
).dt.strftime("%d/%m/%Y")

In [47]:
df_range_final = df_range[[
    "Provider",
    "OfferId",
    "MinGlobal",
    "MinRecommended",
    "MaxRecommended",
    "DifferenceMinRecommendMinTheory",
    "EnqueuedTimeSP"
]]

df_range_final.to_csv("DynamicPriceRange.csv", index=False)
print("DynamicPriceRange.csv created!")

DynamicPriceRange.csv created!
